# Quixer 量子 Transformer 完全教程

> **从 GPT 到量子 Transformer：一步一步理解、实现并运行 Quixer**

---

## 关于本教程

### 目标受众

本教程面向具备以下背景的读者：

- **熟悉 GPT/Transformer 架构**：理解自注意力（Self-Attention）、多头注意力（Multi-Head Attention）、前馈网络（Feed-Forward Network）、层归一化（Layer Normalization）、位置编码（Positional Encoding）、残差连接（Residual Connection）等核心组件。
- **具备量子计算基础概念**：理解量子比特（Qubit）、叠加态（Superposition）、酉变换（Unitary Transformation）、测量（Measurement）、量子门（Pauli 门、旋转门等）的基本含义。

### 你将学到什么

完成本教程后，你将能够：

1. 解释 Quixer 与经典 GPT/Transformer 的本质区别
2. 理解线性酉组合（LCU）和量子奇异值变换（QSVT）的数学原理
3. 掌握 Quixer 的完整架构：从词嵌入到最终输出的每一步
4. 阅读并理解 Quixer 的 PyTorch 经典模拟源码
5. 在自己的机器上搭建环境、训练并运行 Quixer
6. 评估 Quixer 的当前局限性和未来发展方向

### 教程结构

| 章节 | 内容 | 建议时间 |
|------|------|----------|
| 第一章 | GPT/Transformer 架构回顾 | 20 分钟 |
| 第二章 | 量子计算基础回顾 | 30 分钟 |
| 第三章 | Quixer 宏观概览 | 15 分钟 |
| 第四章 | 块编码（Block Encoding） | 30 分钟 |
| 第五章 | 线性酉组合（LCU） | 45 分钟 |
| 第六章 | 量子奇异值变换（QSVT） | 45 分钟 |
| 第七章 | Quixer 完整架构详解 | 40 分钟 |
| 第八章 | 源码深度解析 | 50 分钟 |
| 第九章 | 环境搭建与动手运行 | 40 分钟 |
| 第十章 | 实验结果与性能分析 | 20 分钟 |
| 第十一章 | 局限性与开放问题 | 20 分钟 |
| 第十二章 | 后续研究方向与生态 | 20 分钟 |

---

## 第一章：GPT/Transformer 架构回顾

> **目标**：建立一个清晰的经典 Transformer 心智模型，为理解 Quixer 的『量子替代方案』铺路。

### 1.1 核心处理管线

GPT 系列模型的本质是一个**下一令牌预测器**（Next-Token Predictor）。给定一段序列 $[w_1, w_2, ..., w_n]$，模型预测第 `n+1` 个令牌的概率分布。其核心处理管线如下：

```mermaid
---
config:
  theme: base
  themeVariables:
    primaryColor: '#FAFAFC'
    primaryTextColor: '#222222'
    primaryBorderColor: '#28749A'
    lineColor: '#28749A'
    secondaryColor: '#E9E9E9'
    tertiaryColor: '#FFFFFF'
---
flowchart TD
    INPUT["输入令牌<br/>[w_1, ..., w_n]"]
    
    INPUT --> EMB["词嵌入 (Embedding)<br/>→ [n, d_model] 实数矩阵"]
    
    EMB --> PE["位置编码 (Positional Encoding)<br/>→ 注入序列位置信息"]
    
    PE --> BLOCK
    
    subgraph BLOCK["Transformer 块 (×N 层)"]
        direction TB
        ATTN["多头自注意力<br/>(Multi-Head Self-Attention)<br/>+ 残差连接 + 层归一化"]
        FFN["前馈网络<br/>(Feed-Forward Network)<br/>+ 残差连接 + 层归一化"]
        ATTN --> FFN
    end
    
    BLOCK --> PROJ["输出投影 (Output Projection)<br/>→ [n, vocab_size] logits"]
    
    PROJ --> SM["Softmax<br/>→ 下一令牌的概率分布"]
```

### 1.2 最关键的部分：点积自注意力

自注意力是 Transformer 的**灵魂**。其核心公式为：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

展开来解释：

1. **线性投影**：输入 $X \in \mathbb{R}^{n \times d}$ 通过三个可训练的权重矩阵 $W_Q$, $W_K$, $W_V$ 投影为查询（Query）、键（Key）、值（Value）：
   $$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

2. **注意力分数**：计算 Query 和 Key 之间的相似度（点积），得到 `n × n` 的注意力矩阵：
   $$A = \frac{QK^T}{\sqrt{d_k}}$$

3. **Softmax 归一化**：将每行的相似度转换为概率分布（和为 1）：
   $$\text{softmax}(A)_{ij} = \frac{\exp(A_{ij})}{\sum_k \exp(A_{ik})}$$

4. **加权聚合**：用注意力权重对 Value 矩阵加权求和，实现**令牌混合**（Token Mixing）：
   $$\text{Output} = \text{softmax}(A) \cdot V$$

### 1.3 自注意力的本质：令牌混合

从更高层次看，自注意力的本质是**令牌混合**（Token Mixing）：

- 输入是 `n` 个令牌的表示
- 每个令牌的输出是**所有输入令牌表示的加权平均**
- 权重由令牌之间的『相关性』（Query-Key 相似度）决定
- 这是**内容相关**的混合：混合权重依赖于输入内容本身

### 1.4 为什么量子计算可能提供更好的混合方式？

经典自注意力有两个著名的计算瓶颈：

1. **二次复杂度**：注意力矩阵是 `n × n` 的，计算和存储复杂度都是 `O(n²)`，对长序列不友好
2. **参数规模**：$W_Q$, $W_K$, $W_V$, $W_O$ 等权重矩阵占据大量参数

量子计算提供了几种潜在的替代方案：

- **酉变换**天然保持范数，可能实现更稳定的信息传播
- **量子叠加**可以在指数大的希尔伯特空间中编码信息
- **量子干涉**可以实现经典计算难以表达的函数
- **量子原语**（如 QSVT）可以直接对矩阵施加多项式变换

> **关键思想**：Quixer 不是对『点积自注意力』做量子化改造，而是用全新的量子令牌混合机制来替代整个自注意力模块。

---

## 第二章：量子计算基础回顾

> **目标**：快速回顾理解 Quixer 所需的量子计算核心概念。

### 2.1 量子比特与态矢量

**单量子比特**的态可以写作：

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad |\alpha|^2 + |\beta|^2 = 1, \quad \alpha, \beta \in \mathbb{C}$$

其中：
- $|0\rangle = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$，$|1\rangle = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$
- $\alpha$ 和 $\beta$ 称为**振幅**（Amplitude）
- $|\alpha|^2$ 是测量得到 0 的概率

**多量子比特**系统的态是各量子比特态的张量积。例如，2 量子比特系统：

$$|\psi\rangle = \alpha_{00}|00\rangle + \alpha_{01}|01\rangle + \alpha_{10}|10\rangle + \alpha_{11}|11\rangle$$

$q$ 个量子比特的态矢量在 $\mathbb{C}^{2^q}$ 空间中——**维数随量子比特数指数增长**。

### 2.2 酉变换与量子门

量子计算通过**酉变换**（Unitary Transformation）操作量子态。酉矩阵 $U$ 满足 $U^\dagger U = I$。酉变换的性质是可逆且保范数（$\|U|\psi\rangle\| = \||\psi\rangle\|$）。

常用量子门：

| 门 | 矩阵 | 说明 |
|----|------|------|
| Pauli-X | $$\begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$$ | 量子版 NOT |
| Pauli-Y | $$\begin{bmatrix} 0 & -i \\ i & 0 \end{bmatrix}$$ | |
| Pauli-Z | $$\begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}$$ | 相位翻转 |
| $$R_X(\theta)$$ | $$e^{-i\theta X} = \begin{bmatrix} \cos(\theta) & -i\sin(\theta) \\ -i\sin(\theta) & \cos(\theta) \end{bmatrix}$$ | 绕 X 轴旋转 |
| $$R_Y(\theta)$$ | $$e^{-i\theta Y} = \begin{bmatrix} \cos(\theta) & -\sin(\theta) \\ \sin(\theta) & \cos(\theta) \end{bmatrix}$$ | 绕 Y 轴旋转 |
| $$R_Z(\theta)$$ | $$e^{-i\theta Z} = \begin{bmatrix} e^{-i\theta} & 0 \\ 0 & e^{i\theta} \end{bmatrix}$$ | 绕 Z 轴旋转 |
| CNOT | $$\begin{bmatrix} 1&0&0&0 \\ 0&1&0&0 \\ 0&0&0&1 \\ 0&0&1&0 \end{bmatrix}$$ | 受控非门 |

### 2.3 受控门

**受控门** `CU` 的行为是：
- 当控制量子比特为 $|0\rangle$ 时，目标量子比特不变
- 当控制量子比特为 $|1\rangle$ 时，对目标量子比特施加 $U$

由于量子力学是线性的，当控制量子比特处于叠加态时，受控门产生纠缠：

$$CU(\alpha|0\rangle + \beta|1\rangle) \otimes |\psi\rangle = \alpha|0\rangle \otimes |\psi\rangle + \beta|1\rangle \otimes U|\psi\rangle$$

### 2.4 测量与期望值

测量量子态 $|\psi\rangle$ 中某个可观测量（Observable）$O$（$O$ 是厄米矩阵，$O^\dagger = O$）：

$$\langle O \rangle_{|\psi\rangle} = \langle\psi|O|\psi\rangle \in \mathbb{R}$$

例如，测量 Pauli-Z 的期望值给出量子比特偏向 $|0\rangle$ 还是 $|1\rangle$。

### 2.5 参数化量子电路（PQC）

**参数化量子电路**（Parameterized Quantum Circuit, PQC）是量子机器学习的核心构件。PQC 由一系列参数化的量子门（如 $R_X(\theta), R_Y(\phi), R_Z(\omega)$）组成，其中角度参数是可训练的。

**PQC 结构示例**（Ansatz 14）：
```mermaid
---
config:
  theme: base
  themeVariables:
    primaryColor: '#FAFAFC'
    primaryTextColor: '#222222'
    primaryBorderColor: '#28749A'
    lineColor: '#28749A'
    secondaryColor: '#E9E9E9'
    tertiaryColor: '#FFFFFF'
---
flowchart LR
    B1["<b>第 1 块：RY</b><br/>(前向环形)<br/>
    RY(θ₀)
    RY(θ₁)
    ⋮
    RY(θ<sub>q−1</sub>)"]

    B2["<b>第 2 块：CRX</b><br/>(反向环形)<br/>
    CRX(φ₀)
    CRX(φ₁)
    ⋮
    CRX(φ<sub>q−1</sub>)"]

    B3["<b>第 3 块：RY</b><br/><br/>
    RY(θ′₀)
    RY(θ′₁)
    ⋮
    RY(θ′<sub>q−1</sub>)"]

    B4["<b>第 4 块：CRX</b><br/><br/>
    CRX(φ′₀)
    CRX(φ′₁)
    ⋮
    CRX(φ′<sub>q−1</sub>)"]

    B1 --> B2 --> B3 --> B4

    B1:::ry
    B2:::crx
    B3:::ry
    B4:::crx

    classDef ry fill:#f8fbfd,stroke:#28749A,stroke-width:1.5px,color:#222
    classDef crx fill:#f5f5f5,stroke:#28749A,stroke-width:1.5px,color:#222
```

### 2.6 后选择（Postselection）

**后选择**是量子计算中的一个重要技术：只保留那些某个辅助测量结果为特定值的执行分支，丢弃其他分支。

在电路图中，后选择通常表示为控制量子比特末端的 $\langle 0|$ 标记：

```
|0⟩ ──[U_PREP]──●──[U_PREP†]── ⟨0|
                │
|ψ⟩ ───────────[U_SEL]─────────
```

这意味着：只有当控制量子比特的最终测量结果是 $|0\rangle$ 时，我们才保留这次执行的结果。后选择使得我们可以在酉电路框架中实现非酉操作（如矩阵的非酉线性组合）。

### 2.7 关键概念：块编码（Block Encoding）

**块编码**是理解整个 Quixer 架构的**最重要的前置概念**。我们将其单独放在第四章深入讲解。这里先给出直觉：

> 块编码是一种用**更大的酉矩阵**来『编码』一个**任意（可能非酉）矩阵**的技术。具体来说，如果对某个矩阵 $M$ 存在酉矩阵 $U_M$ 使得 $M$ 是 $U_M$ 的左上角子矩阵，即 $M = (\langle 0| \otimes I) U_M (|0\rangle \otimes I)$，我们就说 $U_M$ 是 $M$ 的一个块编码。

---

## 第三章：Quixer 宏观概览

> **目标**：在深入技术细节之前，先建立对 Quixer 的全局理解。

### 3.1 一句话定义

**Quixer**（Quantum Transformer）是 Quantinuum 于 2024 年 6 月发布的**量子 Transformer 模型**。其核心创新在于：用两种量子计算原语——**线性酉组合（LCU）**和**量子奇异值变换（QSVT）**——完全替代了经典 Transformer 的点积自注意力机制。

### 3.2 Quixer 与 GPT 的本质区别

| 维度 | GPT / 经典 Transformer | Quixer |
|------|---------------------|--------|
| 令牌混合方式 | $\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$ 点积注意力 | LCU：酉变换的加权叠加 |
| 非线性来源 | ReLU / GELU 前馈网络 | QSVT：矩阵的多项式变换 |
| 表示空间 | $\mathbb{R}^d$（实向量空间） | $\mathbb{C}^{2^q}$（希尔伯特空间） |
| 参数形式 | 实矩阵 $W_Q$, $W_K$, $W_V$ | 复数权重 + 量子门旋转角度 |
| 混合机制 | 内容相关（$QK^\top$ 相似度） | 参数化酉电路（结构先验） |

### 3.3 Quixer 的完整处理管线

```mermaid
---
config:
  theme: base
  themeVariables:
    primaryColor: '#FAFAFC'
    primaryTextColor: '#222222'
    primaryBorderColor: '#28749A'
    lineColor: '#28749A'
    secondaryColor: '#E9E9E9'
    tertiaryColor: '#FFFFFF'
---
flowchart TD

    INPUT["输入令牌
    [w<sub>1</sub>, w<sub>2</sub>, &hellip;, w<sub>n</sub>]
    （窗口大小 = n）"]

    S1["<b>步骤 1：经典词嵌入</b><br/>
    Embedding(w<sub>i</sub>) &rarr;
    w<sub>i</sub> &isin; &Ropf;<sup>d<sub>emb</sub></sup>"]

    S2["<b>步骤 2：角度参数化</b><br/>
    &theta;<sub>i</sub> = W<sub>E</sub> &middot; w<sub>i</sub>
    （线性变换）<br/>
    将这些角度作为 PQC 的旋转门参数"]

    S3["<b>步骤 3：酉令牌嵌入</b>
    （Unitary Token Embedding）<br/>
    U<sub>i</sub> = PQC(&theta;<sub>i</sub>)<br/>
    每个令牌获得一个酉矩阵表示
    <small>经典模拟中：量子电路作用于 |0<sup>q</sup>&rang;</small>"]

    S4["<b>步骤 4：LCU 令牌混合</b><br/>
    M = &sum;<sub>j=0</sub><sup>n&minus;1</sup> b<sub>j</sub> &middot; U<sub>j</sub><br/>
    其中 b<sub>j</sub> = e<sup>i&gamma;<sub>j</sub></sup> |a<sub>j</sub>|<sup>2</sup>
    （可训练的复数权重）<br/>
    满足 &sum;<sub>j</sub> |b<sub>j</sub>| = 1"]

    S5["<b>步骤 5：QSVT 非线性多项式变换</b><br/>
    P(M) = c<sub>d</sub>M<sup>d</sup>
    + c<sub>d&minus;1</sub>M<sup>d&minus;1</sup>
    + &hellip;
    + c<sub>1</sub>M
    + c<sub>0</sub>I<br/>
    多项式系数 c<sub>k</sub> 是可训练的"]

    S6["<b>步骤 6：量子前馈网络</b>
    （Quantum Feed-Forward）<br/>
    再次应用 PQC
    共享参数的单层 Ansatz 14"]

    S7["<b>步骤 7：测量</b><br/>
    在所有量子比特上测量
    Pauli-X、Y、Z 的期望值<br/>
    &rarr; 3q 个实数测量值"]

    S8["<b>步骤 8：经典输出投影</b><br/>
    f<sub>out</sub> : &Ropf;<sup>3q</sup> &rarr;
    &Ropf;<sup>vocab_size</sup><br/>
    两层 MLP：Linear &rarr; ReLU &rarr; Linear"]

    OUTPUT["输出 logits
    &darr;
    Softmax
    &darr;
    下一令牌预测"]

    INPUT --> S1 --> S2 --> S3 --> S4 --> S5 --> S6 --> S7 --> S8 --> OUTPUT

    classDef input fill:#FFFFFF,stroke:#28749A,stroke-width:1.5px,color:#222222
    classDef classical fill:#FAFAFC,stroke:#28749A,stroke-width:1.5px,color:#222222
    classDef quantum fill:#E9E9E9,stroke:#28749A,stroke-width:1.5px,color:#222222
    classDef output fill:#FFFFFF,stroke:#28749A,stroke-width:1.5px,color:#222222

    class INPUT,OUTPUT input
    class S1,S2,S8 classical
    class S3,S4,S5,S6,S7 quantum
```

### 3.4 为什么要这样设计？

Quixer 的设计哲学（引自原论文第 2.1 节）：

> "Our focus in this work is not to quantise the dot product self-attention, but instead propose a novel form of token mixing built from quantum primitives."

翻译：**我们不试图用量子电路逐模块地『翻译』经典自注意力，而是从量子计算的基本原语出发，构建一种全新的令牌混合范式。**

这意味着 Quixer 的思路是**量子原生**（Quantum-Native）的：从量子计算能高效做什么出发，而不是从经典 Transformer 有什么出发。

### 3.5 重要声明：经典模拟

Quixer 的官方 GitHub 仓库（github.com/Quantinuum/Quixer）提供的**仅仅是经典模拟实现**——所有量子操作都用 PyTorch 张量在 CPU/CUDA 上模拟，**没有真正的量子硬件后端**。这就像在普通计算机上运行量子电路的仿真器。截至 2026 年 5 月，虽然 Quantinuum 宣称已将 Quixer 部署到其 H 系列量子硬件上，但该声明缺乏独立的第三方验证。

---

## 第四章：块编码（Block Encoding）

> **目标**：深入理解块编码——这是 LCU 和 QSVT 的基础。

### 4.1 动机：为什么需要块编码？

量子计算的基本操作是**酉变换**。但机器学习中涉及的许多矩阵操作——如线性组合、非线性激活——本质上是**非酉的**。

块编码（Block Encoding）是一个巧妙的桥接技术：它允许我们用**酉矩阵的左上角子矩阵**来表示任意（可能非酉的）矩阵。

### 4.2 形式化定义

给定一个矩阵 $M$（不一定方阵，不一定酉），如果存在酉矩阵 $U$ 使得：

$$M = (\langle 0^a| \otimes I) \; U \; (|0^a\rangle \otimes I)$$

其中：
- $a$ 是辅助量子比特的数量
- $|0^a\rangle$ 是 $a$ 个辅助量子比特全部处于 $|0\rangle$ 的状态
- 等式右端表示：将 $U$ 作用于 $|0^a\rangle \otimes |\psi\rangle$，然后在辅助量子比特上后选择 $|0^a\rangle$

则称 $U$ 是 $M$ 的一个**块编码**，$M$ 是 $U$ 的左上角子矩阵。

### 4.3 直观理解

想象 $U$ 是一个分块矩阵：

$$U = \begin{bmatrix} M & \cdot \\ \cdot & \cdot \end{bmatrix}$$

当我们：
1. 把辅助量子比特初始化为 $|0^a\rangle$
2. 把数据量子比特初始化为 $|\psi\rangle$
3. 对整个系统施加 $U$
4. 测量辅助量子比特，只保留结果为 $|0^a\rangle$ 的分支

那么数据量子比特经历的**有效变换**正好是 $M$：

$$|\psi\rangle \xrightarrow{\text{block-encoded } U} M|\psi\rangle \quad (\text{归一化前})$$

### 4.4 块编码的经典类比

这个思想并不是量子力学的专利。在经典计算中也有类似的做法：

- **经典类比 1（嵌入矩阵）**：如果你想用正交矩阵表示非正交的线性变换，可以将其嵌入一个更高维的正交矩阵中。

- **经典类比 2（条件执行）**：就像程序中的条件分支——
  ```python
  if flag == 0:
      result = M @ vector  # 这种『如果前置条件满足就执行 M』的逻辑
  ```
  块编码用叠加态的线性性质将这种『条件执行』延拓为对所有分支同时运算。

- **经典类比 3（分块矩阵）**：就像你可以在一个更大的矩阵中『隐藏』一个小的子矩阵，块编码在一个更大的酉矩阵中『隐藏』了非酉的 $M$，而量子力学的线性性和后选择机制将这种『隐藏』变成了可执行的电路。

### 4.5 块编码的关键性质

1. **谱性质**：如果 $M$ 的奇异值在 $[0, 1]$ 内，块编码存在且可用标准技术构造
2. **多项式变换**：如果我们能对块编码 $U_M$ 做某些操作，就能对整个 $M$ 做相应的矩阵多项式变换——这就是 QSVT 的核心思想
3. **线性组合**：如果我们能构造一系列矩阵的块编码，就能构造它们线性组合的块编码——这就是 LCU 的核心思想

---

## 第五章：线性酉组合（LCU）

> **目标**：掌握 LCU 的完整数学原理和电路实现。

### 5.1 回到经典 Transformer 的令牌混合

在经典 Transformer 中，每个令牌的输出是所有令牌 Value 的加权平均：

$$\text{Output}_i = \sum_{j=1}^n \alpha_{ij} V_j, \quad \sum_j \alpha_{ij} = 1$$

权重 $\alpha_{ij}$ 是 softmax 归一化的注意力分数。

### 5.2 LCU 的量子类比

Quixer 用**酉变换的加权叠加**来类比这种混合：

$$M = \sum_{j=0}^{n-1} b_j \, U_j$$

其中：
- $U_j \in \mathbb{C}^{2^q \times 2^q}$ 是第 $j$ 个令牌对应的**酉矩阵**（由参数化量子电路生成）
- $b_j \in \mathbb{C}$ 是**可训练的复数权重**
- $n$ 是上下文窗口大小
- 满足归一化条件：$\sum_j |b_j| = 1$

> **直觉类比**：经典注意力是 $\sum \alpha_{ij} V_j$（向量的加权平均）；LCU 令牌混合是 $\sum b_j U_j$（酉矩阵的复数加权线性组合）。$V_j$ 是向量，$U_j$ 是矩阵（更强大的表示）。

### 5.3 SELECT-PREPARE 电路：从数学到电路

LCU 的关键问题是：**如何在量子电路中实现这种『酉变换的非酉组合』？** 答案是通过 SELECT-PREPARE 电路架构。

#### 准备阶段（PREPARE）

首先，构造一个酉变换 $U_{\text{PREP}}$ 作用于控制寄存器（大小为 $\lceil \log_2 n \rceil$ 个量子比特），将 $|0\ldots 0\rangle$ 准备为所需的系数叠加态：

$$U_{\text{PREP}} |0\rangle = |a\rangle = \sum_{j=0}^{n-1} a_j |j\rangle$$

其中 $a_j$ 是复数振幅，$|j\rangle$ 是控制寄存器的计算基态。

#### 选择阶段（SELECT）

然后，构造一个受控酉变换 $U_{\text{SEL}}$：

$$U_{\text{SEL}} = \sum_{j=0}^{n-1} |j\rangle\langle j| \otimes U_j$$

解读：$U_{\text{SEL}}$ 的每一『块』对应控制寄存器的一个计算基态：
- 当控制寄存器为 $|0\rangle$ 时，对数据寄存器施加 $U_0$
- 当控制寄存器为 $|1\rangle$ 时，对数据寄存器施加 $U_1$
- ...
- 当控制寄存器为 $|n-1\rangle$ 时，对数据寄存器施加 $U_{n-1}$

#### 完整的 LCU 电路 $U_M$

组合这两个操作：

$$U_M = (U_{\text{PREP}}^\dagger \otimes I) \; U_{\text{SEL}} \; (U_{\text{PREP}} \otimes I)$$

然后在控制寄存器上后选择 $|0\rangle$：

$$M = (\langle 0| \otimes I) \; U_M \; (|0\rangle \otimes I)$$

数学上可以证明（见论文附录 A.2 的引理 1）：

$$M = (\langle 0| \otimes I) \; U_M \; (|0\rangle \otimes I) = \sum_{j=0}^{n-1} |a_j|^2 \, U_j$$

### 5.4 为什么要设计成这种形式：直观的数学推导

让我们一步一步追踪态在电路中的演化，理解为什么最终结果正好是 $\sum |a_j|^2 U_j$。

**初始态**：控制寄存器在 $|0\rangle$，数据寄存器在 $|\psi\rangle$：
$$|\Phi_0\rangle = |0\rangle \otimes |\psi\rangle$$

**步骤 1**：应用 $U_{\text{PREP}} \otimes I$。控制寄存器变为叠加态 $|a\rangle = \sum a_j |j\rangle$：
$$|\Phi_1\rangle = \left(\sum_{j=0}^{n-1} a_j |j\rangle\right) \otimes |\psi\rangle$$

**步骤 2**：应用 $U_{\text{SEL}}$。每个 $|j\rangle$ 分支上数据寄存器被施加对应的 $U_j$：
$$|\Phi_2\rangle = \sum_{j=0}^{n-1} a_j |j\rangle \otimes U_j |\psi\rangle$$

**步骤 3**：应用 $U_{\text{PREP}}^\dagger \otimes I$。这『解纠缠』控制寄存器。考虑后选择 $|0\rangle$：
$$|\Phi_3\rangle = |0\rangle \otimes \left(\sum_{j=0}^{n-1} a_j \langle 0|U_{\text{PREP}}^\dagger|j\rangle \cdot U_j |\psi\rangle\right)$$

由于 $U_{\text{PREP}}^\dagger |j\rangle$ 投影到 $\langle 0|$ 给出 $a_j^*$（因为 $U_{\text{PREP}}|0\rangle = \sum a_j|j\rangle$），所以：
$$\langle 0|U_{\text{PREP}}^\dagger|j\rangle = a_j^*$$

因此：
$$|\Phi_3\rangle = |0\rangle \otimes \sum_{j=0}^{n-1} |a_j|^2 \, U_j |\psi\rangle$$

后选择成功概率 $p_{\text{success}} = \|\sum_{j} |a_j|^2 U_j |\psi\rangle\|^2$。

这完美地实现了：**酉矩阵的线性组合，系数为 $|a_j|^2$**（非负实数）。

### 5.5 复数系数的引入

但实数系数 $|a_j|^2$ 的表达力有限！为了实现复数系数 $b_j \in \mathbb{C}$，Quixer 在每个令牌的酉电路前加入了一个相位门：

$$U_j' = e^{i\gamma_j} \cdot U_j$$

这样，最终的有效混合变为：

$$M = \sum_{j=0}^{n-1} \underbrace{e^{i\gamma_j} |a_j|^2}_{b_j \in \mathbb{C}} \, U_j$$

其中 $b_j = e^{i\gamma_j} |a_j|^2$ 是**可训练的复数系数**，满足 $\sum |b_j| = 1$。

### 5.6 后选择概率与资源开销

虽然 LCU 电路使用了辅助量子比特和测量的组合实现了看似非酉的操作，但这并非没有代价：

- **后选择成功率** $p_{\text{success}} = \|\sum b_j U_j|\psi\rangle\|^2$ 并不总是 1
- 在实际量子硬件上，你需要执行 $O(1/p_{\text{success}})$ 次测量才能获得一个有效样本
- 当系数 $|b_j|$ 分布极不均匀（某个系数远大于其他系数）时，成功率可能接近 1；当系数分布均匀时，成功率可能较低

### 5.7 LCU 与经典注意力的深层对比

| 属性 | 经典注意力 | LCU 令牌混合 |
|------|-----------|-------------|
| 混合对象 | Value 向量 $V_j$ | 酉矩阵 $U_j$ |
| 混合权重 | $$\operatorname{softmax}(QK^\top/\sqrt{d_k})$$（内容相关） | $b_j$（可训练参数） |
| 权重空间 | 实数、正数、和为 1 | 复数、L1 范数为 1 |
| 复杂度 | $O(n^2d)$（经典计算） | $$O(n \cdot 4^q)$$（经典模拟）或 $$O(n \cdot \mathrm{poly}(q))$$（量子实现） |
| 信息传播 | 点积交互 | 酉变换的干涉效应 |

### 5.8 一个具体的数值例子

假设窗口大小 $n = 3$，意味着我们有 3 个令牌，每个令牌有一个对应的酉矩阵 $U_0, U_1, U_2$。

```
# 经典注意力（抽象表示）
Q, K, V = X @ W_Q, X @ W_K, X @ W_V
attn = softmax(Q @ K.T / sqrt(d_k))
output = attn @ V  # 每个输出是 V 的加权平均

# Quixer 的 LCU（抽象表示）
U = [U_0, U_1, U_2]          # 每个令牌的酉矩阵
b = [0.5+0.0j, 0.3+0.1j, 0.1+0.0j]  # 可训练复数权重
M = b[0]*U[0] + b[1]*U[1] + b[2]*U[2]  # 酉矩阵的线性组合
output_state = M @ |ψ_in⟩     # 混合算子作用于量子态
```

注意关键区别：
- 经典注意力中，softmax 的归一化是对『行』施加的（每个查询的输出权重之和为 1）
- LCU 中，归一化是对整个系数向量施加的（$\sum|b_j|=1$）
- 这意味着 LCU 对所有令牌使用**同一组全局混合权重**，而不是每个令牌有独立的注意力分布——这是一种**全局令牌混合**而非**内容相关的自适应混合**

---

## 第六章：量子奇异值变换（QSVT）

> **目标**：掌握 QSVT 的原理，理解它如何为量子电路提供非线性。

### 6.1 动机：量子电路缺少非线性

在经典深度学习中，网络成功的一个关键因素是**非线性激活函数**（ReLU、GELU、tanh 等）。没有非线性，多层网络等价于单层线性变换。

但量子力学本质上是**线性的**——薛定谔方程是线性方程，量子门是酉（线性）变换。那么如何在量子电路中引入非线性？

**QSVT 提供了一种方法**：对块编码的矩阵施加**多项式变换**。多项式是非线性的（在矩阵意义上：$M^2 \neq cM$），因此 QSVT 为量子计算提供了一种『矩阵级』的非线性机制。

### 6.2 QSVT 的数学定义

给定一个矩阵 $M$ 和它的块编码酉矩阵 $U_M$，QSVT 允许我们构造一个电路来实现 $M$ 的任意**奇偶性匹配的多项式**：

$$P_c(M) = c_d M^d + c_{d-1} M^{d-1} + \cdots + c_1 M + c_0 I$$

条件是：
1. **有界性**：$|P_c(x)| \leq 1$ 对所有 $x \in [-1, 1]$ 成立（多项式在 $[-1,1]$ 上有界）
2. **奇偶性约束**：$\text{parity}(P_c) = d \bmod 2$（多项式的奇偶性必须与其次数一致）

### 6.3 QSVT 电路结构

QSVT 的核心思想是：**交替应用 $U_M$、$U_M^\dagger$ 和投影算子 $\Pi_\phi$**。

#### 奇数次多项式（$d$ 为奇数）

$$\Pi_{\phi_1} U_M \left[ \prod_{k=1}^{(d-1)/2} \Pi_{\phi_{2k}} U_M^\dagger \Pi_{\phi_{2k+1}} U_M \right]$$

#### 偶数次多项式（$d$ 为偶数）

$$\prod_{k=1}^{d/2} \Pi_{\phi_{2k-1}} U_M^\dagger \Pi_{\phi_{2k}} U_M$$

其中：
- $\Pi_\phi = e^{i\phi(2\Pi - I)} = e^{i\phi Z}$ 是辅助量子比特上的相位旋转
- $\{\phi_1, \phi_2, \ldots, \phi_d\}$ 是与多项式系数 $\{c_0, c_1, \ldots, c_d\}$ 一一对应的相位角
- $U_M$ 块编码 $M$，$U_M^\dagger$ 块编码 $M^\dagger$

### 6.4 相位角度 {φ_k} 的计算

从多项式系数 $\{c_k\}$ 到 QSVT 相位角度 $\{\phi_k\}$ 的映射是**非平凡**的——需要通过数值算法计算。这个算法在量子计算领域是标准化的，常用工具（如 QSPPACK、pyqsp）提供实现。

**关键理解**：在 Quixer 的经典模拟中，由于模拟的是『QSVT 的数学效果』而非『QSVT 电路』，因此**不需要计算相位角度**——直接用多项式系数 $c_k$ 做代数计算即可（详见第八章源码解析）。

### 6.5 奇偶性约束的解除

QSVT 的核心约束是：多项式的奇偶性必须等于其次数的奇偶性。即：

- 如果 $d = 3$（奇数），那么 $P_c(x)$ 只能包含奇次项：$c_3 x^3 + c_1 x$（$c_2 = c_0 = 0$）
- 如果 $d = 2$（偶数），那么 $P_c(x)$ 只能包含偶次项：$c_2 x^2 + c_0 I$（$c_1 = 0$）

**如何获得任意奇偶性的多项式？** 论文提供了一种技巧：引入一个额外的辅助量子比特来控制两种奇偶性多项式的线性组合：

$$P_{\text{arbitrary}}(M) = \alpha \cdot P_{\text{even}}(M) + \beta \cdot P_{\text{odd}}(M)$$

这本质上是用 LCU 的方法组合两个 QSVT 电路——量子原语的组合！

### 6.6 QSVT 与经典非线性激活的类比

| 经典 | 量子（QSVT） |
|------|-------------|
| σ(Wx) = ReLU(Wx) | P(M) = c₀I + c₁M + c₂M² + ... |
| 逐元素非线性 | 矩阵级多项式非线性 |
| 作用于向量 | 作用于整个矩阵 |
| 简单的代数操作 | 需要块编码 + 交替门电路 |
| 计算开销可忽略 | 多项式次数 d 倍于一次块编码调用 |

> **核心理解**：QSVT 的『非线性』不是经典意义上的逐元素非线性（如对每个元素取 max(0, x)），而是**矩阵多项式**带来的非线性。$M^2 \neq cM$（除非 M 是数量矩阵的倍数），所以即使二次多项式 $M^2$ 也是『矩阵非线性』的。

### 6.7 一个简单的例子：d=2 的 QSVT

假设 $M$ 是一个 2×2 的 Hermitian 矩阵：

$$M = \begin{bmatrix} 0.5 & 0.2 \\ 0.2 & -0.3 \end{bmatrix}$$

二次 QSVT 的变换（$d=2$，$c_0=0.1, c_1=0, c_2=0.9$，满足偶次约束）：

$$P(M) = 0.1 I + 0.9 M^2$$

手动计算：
$$M^2 = M \cdot M = \begin{bmatrix} 0.29 & 0.04 \\ 0.04 & 0.13 \end{bmatrix}$$

$$P(M) = 0.1\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} + 0.9\begin{bmatrix} 0.29 & 0.04 \\ 0.04 & 0.13 \end{bmatrix} = \begin{bmatrix} 0.361 & 0.036 \\ 0.036 & 0.217 \end{bmatrix}$$

注意 $P(M)$ 与 $M$ 的关系不是简单的缩放——这就是矩阵多项式的『非线性』效果。它改变了矩阵的谱（特征值分布），这与经典 FFN 中 ReLU 改变向量激活模式的效果类似。

---

## 第七章：Quixer 完整架构详解

> **目标**：逐步骤理解 Quixer 从输入到输出的完整数据流。

### 7.1 架构全景图

**Quixer 架构全景图**
```mermaid
---
config:
  theme: base
  themeVariables:
    primaryColor: '#FAFAFC'
    primaryTextColor: '#222222'
    primaryBorderColor: '#28749A'
    lineColor: '#28749A'
    secondaryColor: '#E9E9E9'
    tertiaryColor: '#FFFFFF'
---
flowchart TD

    INPUT["输入序列：
    [The, cat, sat, on, mat, &lt;eos&gt;]"]

    S1["<b>1. TOKEN EMBEDDING</b><br/>
    w<sub>i</sub> &rarr; Embedding(w<sub>i</sub>) &isin; ℝ<sup>d<sub>emb</sub></sup>
    每个令牌映射到 d<sub>emb</sub> 维实向量"]

    S2["<b>2. ANGLE PARAMETRIZATION</b><br/>
    &theta;<sub>i</sub> = Dropout(W<sub>E</sub> &middot; w<sub>i</sub>)
    实向量 &rarr; PQC 旋转角度
    &theta;<sub>i</sub> &isin; ℝ<sup>n<sub>pqc_params</sub></sup>"]

    S3["<b>3. UNITARY TOKEN EMBEDDING</b><br/>
    对每个令牌 j：
    • 初始化为 |0&hellip;0&rang;
    • 施加 PQC(&theta;<sub>j</sub>) &rarr; 获得 |&psi;<sub>j</sub>&rang;
    • |&psi;<sub>j</sub>&rang; 编码了令牌 j 的量子表示"]

    S4["<b>4. LCU TOKEN MIXING</b><br/>
    M = &sum;<sub>j=0</sub><sup>n&minus;1</sup> b<sub>j</sub> &middot; U<sub>j</sub>
    b<sub>j</sub> = e<sup>i&gamma;<sub>j</sub></sup> |a<sub>j</sub>|<sup>2</sup> &isin; ℂ
    &sum; |b<sub>j</sub>| = 1<br/>
    在经典模拟中的等价操作：
    对 |&psi;<sub>mix</sub>&rang; = M|0&hellip;0&rang; 直接计算"]

    S5["<b>5. QSVT NONLINEAR TRANSFORM</b><br/>
    P(M)|0&hellip;0&rang;
    = (&sum; c<sub>k</sub>M<sup>k</sup>) |0&hellip;0&rang;<br/>
    多项式次数 = qsvt_polynomial_degree
    系数 c<sub>k</sub> 可训练"]

    S6["<b>6. L2 NORMALIZATION</b><br/>
    |&psi;<sub>norm</sub>&rang; =
    |&psi;&rang; / ‖ |&psi;&rang; ‖"]

    S7["<b>7. QUANTUM FEED-FORWARD</b><br/>
    单层 Ansatz 14 PQC
    |&psi;<sub>out</sub>&rang; =
    U<sub>FF</sub>(&theta;<sub>shared</sub>) |&psi;<sub>norm</sub>&rang;"]

    S8["<b>8. MEASUREMENT</b><br/>
    所有量子比特上测量：
    &lang;X<sub>i</sub>&rang;、&lang;Y<sub>i</sub>&rang;、&lang;Z<sub>i</sub>&rang;
    &nbsp; i = 0&hellip;q&minus;1
    &rarr; 3q 个实数期望值"]

    S9["<b>9. CLASSICAL OUTPUT HEAD</b><br/>
    f<sub>out</sub> : ℝ<sup>3q</sup> &rarr; ℝ<sup>vocab_size</sup><br/>
    Linear(3q &rarr; d<sub>emb</sub>) &rarr; ReLU
    &rarr; Linear(d<sub>emb</sub> &rarr; vocab_size)"]

    OUTPUT["输出：词汇表大小的 logits
    &rarr; 下一令牌概率"]

    INPUT --> S1 --> S2 --> S3 --> S4 --> S5 --> S6 --> S7 --> S8 --> S9 --> OUTPUT

    classDef input fill:#FFFFFF,stroke:#28749A,stroke-width:1.5px,color:#222222
    classDef classical fill:#FAFAFC,stroke:#28749A,stroke-width:1.5px,color:#222222
    classDef quantum fill:#E9E9E9,stroke:#28749A,stroke-width:1.5px,color:#222222
    classDef output fill:#FFFFFF,stroke:#28749A,stroke-width:1.5px,color:#222222

    class INPUT,OUTPUT input
    class S1,S2,S9 classical
    class S3,S4,S5,S6,S7,S8 quantum
```

### 7.2 步骤详解

#### 步骤 1-2：经典词嵌入 + 角度参数化

这是模型中最『经典』的部分。输入令牌索引通过标准的 `nn.Embedding` 映射为 dense 向量，然后通过一个线性层映射为 PQC 的旋转参数：

```python
# 伪代码
x: [batch_size, n_tokens]   # 令牌索引
emb = embedding(x)           # [batch_size, n_tokens, d_emb]
angles = linear(dropout(emb))  # [batch_size, n_tokens, n_pqc_params]
```

其中 `n_pqc_params = 4 * n_qubits * n_ansatz_layers`（Ansatz 14 每个量子比特每层 4 个旋转参数）。

#### 步骤 3：酉令牌嵌入

对于每个令牌 $j$，用其角度 $\theta_j$ 参数化一个 PQC（Ansatz 14），将其作用于初始态 $|0\ldots0\rangle$：

$$|\psi_j\rangle = \text{PQC}(\theta_j) |0\ldots0\rangle$$

在经典模拟中，这意味着对 $|0\ldots0\rangle$（即振幅向量 `[1, 0, 0, ..., 0]`）施加一系列旋转门和受控旋转门，得到 $2^q$ 维的复数态矢量。

```python
# 伪代码（经典模拟中的实际计算）
initial_state = torch.zeros(batch_size, 2**n_qubits, dtype=torch.complex64)
initial_state[:, 0] = 1.0 + 0.0j  # |0...0⟩

# 对每个令牌应用 PQC
states = apply_pqc(initial_state, angles)  # [batch_size, n_tokens, 2**n_qubits]
```

Ansatz 14 的结构（每层）：

```
┌─── RY(θ_0) ───●─── RY(θ'_0) ────────────────────────●───
│               │CRX                                CRX│
├─── RY(θ_1) ───┼─── RY(θ'_1) ───●───────────────────┼───
│               │               │CRX                  │
├─── RY(θ_2) ───┼─── RY(θ'_2) ───┼───●───────────────┼───
│               │               │   │CRX              │
│    ...        ...             ...  ...              ...
│               │               │   │                 │
├─── RY(θ_q-1) ─┼─── RY(θ'_q-1)─┼───┼─── ... ───●────┼───
│               │               │   │          │CRX   │
└───────────────┘───────────────┘───┘──────────┘──────┘
  块1: RY旋转    块2: CRX向前   块3: RY旋转   块4: CRX向后
                  (环形连接)                    (反向环形)
```

#### 步骤 4：LCU 令牌混合

在经典模拟中，这一步直接计算混合：

```python
# 伪代码
# 将初始态复制 n_tokens 份
replicated = initial_state.repeat(1, n_tokens)  # [B, n_tokens * 2**n_qubits]

# 对每份施加对应的 PQC（不同令牌不同参数）
evolved = apply_pqc_all_tokens(replicated, all_angles)
evolved = evolved.reshape(B, n_tokens, 2**n_qubits)

# 加权求和
mixed = einsum("bti,bt->bi", evolved, lcu_coefficients)
```

#### 步骤 5：QSVT 非线性变换

在经典模拟中，QSVT 通过迭代应用 LCU 来构建 $M^k$：

```python
# 伪代码：P(M)|0⟩ = Σ c_k M^k |0⟩
accumulated = c[0] * initial_state     # 常数项 c_0 I
monomial = initial_state

for k in range(1, degree + 1):
    # M^k |0⟩ = M · M^{k-1} |0⟩
    monomial = apply_lcu(monomial, ...)  # 乘以 M
    accumulated += c[k] * monomial       # 累加 c_k M^k |0⟩

# 归一化
result = accumulated / norm(c, ord=1)
```

#### 步骤 6：L2 归一化

LCU + QSVT 的『有效计算』本质上是非酉的（因为是酉矩阵的加权组合，不保持范数），所以需要对量子态做 L2 归一化。这是一个关键操作——在真正的量子硬件中，这种归一化需要利用后选择概率进行，成本较高。经典模拟中直接向量归一化即可。

在 PyTorch 代码中实现为：
```python
state = torch.nn.functional.normalize(state, dim=-1)
```

#### 步骤 7：量子前馈网络

一个**共享参数**的单层 PQC 作用在所有 batch 的量子态上：

```python
# 伪代码
shared_params = ff_params.repeat(1, batch_size)
final_state = apply_single_layer_pqc(normalized_state, shared_params)
```

这里的设计很重要：与令牌嵌入的 PQC（每个令牌独立参数）不同，前馈 PQC 使用**单一参数集**，类似于经典 FFN 对每个位置使用相同的权重矩阵。

#### 步骤 8：测量

对每个量子比特测量 Pauli-X, Y, Z 的三个期望值：

$$e_X^{(i)} = \langle \psi | X_i | \psi \rangle$$
$$e_Y^{(i)} = \langle \psi | Y_i | \psi \rangle$$
$$e_Z^{(i)} = \langle \psi | Z_i | \psi \rangle$$

总共 $3q$ 个实数值。这三个期望值形成了一个完整的单量子比特态的布洛赫球表示（Bloch Vector）。

```python
# 伪代码
measurements = []
for qubit in range(n_qubits):
    measurements.append(measure_x(qubit))  # X 期望值
    measurements.append(measure_y(qubit))  # Y 期望值
    measurements.append(measure_z(qubit))  # Z 期望值
# 结果: [batch_size, 3 * n_qubits]
```

> **为什么测量 X, Y, Z 三者？** 经典上，一个量子比特的密度矩阵可以写作 $\rho = \frac{1}{2}(I + xX + yY + zZ)$，其中 $(x, y, z) = (\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ 完全确定了单量子比特态。因此这三个期望值提供了量子态的完整局部信息。

#### 步骤 9：经典输出投影

$$f_{\text{out}}(x) = W_2 \cdot \text{ReLU}(W_1 \cdot x + b_1) + b_2$$

两层 MLP 将 $3q$ 维的测量向量映射到 `vocab_size` 维的 logits。

```
# 代码
output = Sequential(
    Linear(3*n_qubits → embedding_dimension),
    ReLU(),
    Linear(embedding_dimension → vocabulary_size)
)(measurements)  # [batch_size, vocabulary_size]
```

### 7.3 Quixer 的返回值

Quixer 的 `forward` 方法返回一个元组 `(logits, mean_prob)`：

- **logits**: `[batch_size, vocabulary_size]` — 用于 CrossEntropyLoss 的标准分类 logits
- **mean_prob**: 标量 — QSVT+LCU 之后、量子前馈网络之前的态矢量的 L2 范数的 batch 平均值。论文将其称为『最终概率』（final probability），它反映了通过 LCU+QSVT 混合后有多少『概率质量』得以保留（因为非酉操作会导致范数衰减）。

---

## 第八章：源码深度解析

> **目标**：逐行理解 Quixer 经典模拟实现的源码。

### 8.1 项目结构

```
Quixer/
├── pyproject.toml          # 项目配置和依赖声明
├── run.py                  # 训练入口脚本
├── README.md               # 使用说明
├── LICENSE                 # Apache 2.0
└── quixer/
    ├── quixer_model.py     # Quixer 模型定义
    ├── baseline_models.py  # 经典基线模型 (Transformer, LSTM, FNet)
    └── setup_training.py   # 数据处理和训练循环
```

### 8.2 run.py — 训练入口

In [ ]:
# run.py 的核心逻辑（重构版）

# 超参数定义
quixer_hyperparams = {
    "layers": 3,          # QSVT 多项式次数
    "window": 32,         # 上下文窗口大小
    "epochs": 40,         # 训练轮数
    "dropout": 0.2,       # Dropout 率
    "lr": 0.0001,         # 学习率
    "batch_size": 20,     # 批次大小
    "qubits": 6,          # 量子比特数
    "ansatz_layers": 4,   # PQC 层数
}

# 嵌入维度配置
classical_embedding_dimensions = [96, 128]
quantum_embedding_dimensions = [512]

# 模型注册表
model_map = {
    "Quixer": (quixer_hyperparams, quantum_embedding_dimensions),
    "Transformer": (transformer_hyperparams, classical_embedding_dimensions),
    "LSTM": (lstm_hyperparams, classical_embedding_dimensions),
    "FNet": (fnet_hyperparams, classical_embedding_dimensions),
}

# 主循环
for model_name in requested_models:
    for emb_dim in model_map[model_name].embedding_dims:
        for seed in random_seeds:
            params = {**model_map[model_name].hyperparams, 
                      "model": model_name, 
                      "dimension": emb_dim, 
                      "seed": seed}
            train_evaluate(params)  # 返回测试 loss

**关键观察**：

- Quixer 的嵌入维度固定为 512，而经典基线是 96 或 128——Quixer 需要更大的嵌入维度来提供足够的参数给 PQC
- 对每个配置运行 10 个随机种子，确保统计显著性
- Quixer 默认使用 6 个量子比特、4 层 ansatz、3 次 QSVT 多项式

### 8.3 quixer_model.py — 核心模型

#### 8.3.1 Ansatz 14 电路构建

In [ ]:
def ansatz_14_torchquantum_specification(n_qubits, layers=1):
    """
    构建 Ansatz 14 的电路规格。
    
    每层包含 4 个块，共 4 * n_qubits 个参数：
    - 块1: RY 旋转（所有量子比特）
    - 块2: CRX 向前环形连接 (i → i+1)
    - 块3: RY 旋转（所有量子比特）
    - 块4: CRX 向后环形连接 (i → i-1)
    """
    enc = []
    counter = itertools.count(0)
    
    for _ in range(layers):
        # 块1: RY 旋转
        for i in range(n_qubits):
            enc.append({
                "input_idx": [next(counter)],
                "func": "ry",
                "wires": [i]
            })
        # 块2: CRX 向前环形
        for i in range(n_qubits - 1, -1, -1):  # 反向遍历
            enc.append({
                "input_idx": [next(counter)],
                "func": "crx",
                "wires": [i, (i + 1) % n_qubits]
            })
        # 块3: RY 旋转
        for i in range(n_qubits):
            enc.append({
                "input_idx": [next(counter)],
                "func": "ry",
                "wires": [i]
            })
        # 块4: CRX 向后环形
        for last_qubit in range(n_qubits - 1, -1, -1):
            i = (last_qubit - 1) % n_qubits  # 控制量子比特
            j = last_qubit
            enc.append({
                "input_idx": [next(counter)],
                "func": "crx",
                "wires": [i, j]
            })
    
    return enc

**Ansatz 14 的环形连接设计解读**：

```
n_qubits = 4 时一轮的 CRX 连接：

块2: CRX 向前环形       块4: CRX 向后环形
q0 ←── q3               q0 ──→ q3  
 ↑                         ↓
q1 ←── q0               q1 ──→ q0
 ↑                         ↓
q2 ←── q1               q2 ──→ q1
 ↑                         ↓
q3 ←── q2               q3 ──→ q2
```

这种双向环形连接确保信息可以在所有量子比特之间传播。

#### 8.3.2 LCU 的经典模拟

In [ ]:
def apply_linear_combination_of_unitaries(
    initial_states,          # [batch_size, 2**n_qubits]
    pqc_parameters,          # [batch_size, n_tokens, n_pqc_params]
    parameterized_quantum_circuit,  # TorchQuantum GeneralEncoder
    torchquantum_device,
    n_qubits,
    lcu_coefficients,        # [batch_size, n_tokens], 复数
):
    """
    经典模拟 LCU: M|ψ⟩ = (Σ b_j U_j) |ψ⟩
    
    关键思路：
    1. 将初始态复制 n_tokens 份
    2. 每份分别施加对应令牌的 PQC
    3. 用复数系数加权求和
    """
    batch_size, n_tokens, _ = pqc_parameters.shape
    
    # 复制初始态：每个 batch 元素的态复制 n_tokens 次
    states = initial_states.repeat(1, n_tokens).reshape(-1, 2**n_qubits)
    # states: [batch_size * n_tokens, 2**n_qubits]
    
    # 展平 PQC 参数
    flat_params = pqc_parameters.reshape(-1, pqc_parameters.shape[-1])
    # flat_params: [batch_size * n_tokens, n_pqc_params]
    
    # 将态加载到量子设备中
    torchquantum_device.reset_batch_size(batch_size * n_tokens)
    
    # 对每份态施加对应令牌的 PQC
    parameterized_quantum_circuit(
        torchquantum_device, 
        flat_params.float()
    )
    
    # 提取演化后的态
    evolved = torchquantum_device.get_states_1d()
    evolved = evolved.reshape(batch_size, n_tokens, 2**n_qubits)
    
    # 加权求和: Σ b_j |ψ_j⟩
    mixed = torch.einsum("bti,bt->bi", evolved, lcu_coefficients)
    
    return mixed  # [batch_size, 2**n_qubits]

> **设计洞察**：经典模拟中并没有真正构造 SELECT-PREPARE 电路。代码采用了一种更高效的方式——直接枚举每个酉矩阵 $U_j$，分别计算 $U_j \lvert \psi \rangle$，然后做复数加权的线性求和。这是因为在经典计算机上，显式计算每个 $2^q$ 维态矢量并求和比模拟一个完整的 LCU 电路（需要额外的 $\log_2 n$ 个控制比特）开销更小。在真实量子计算机上，你才需要 SELECT-PREPARE 来利用量子并行性。

#### 8.3.3 QSVT 的经典模拟

In [ ]:
def apply_qsvt_and_lcu(
    initial_states,           # [batch_size, 2**n_qubits]
    pqc_parameters,
    parameterized_quantum_circuit,
    torchquantum_device,
    n_qubits,
    lcu_coefficients,
    qsvt_polynomial_coefficients,  # [degree + 1]
):
    """
    经典模拟 QSVT+LCU: P(M)|ψ⟩ = (Σ c_k M^k) |ψ⟩
    
    通过反复调用 apply_lcu 来构建 M 的幂次。
    """
    # 初始化累加器：c_0 I |ψ⟩ = c_0 |ψ⟩
    accumulated = qsvt_polynomial_coefficients[0] * initial_states
    
    # monomial 追踪 M^{k-1} |ψ⟩
    monomial = initial_states
    
    for k in range(1, len(qsvt_polynomial_coefficients)):
        # M^k |ψ⟩ = M · (M^{k-1} |ψ⟩)
        monomial = apply_lcu(
            monomial, pqc_parameters, 
            parameterized_quantum_circuit,
            torchquantum_device, n_qubits, lcu_coefficients
        )
        
        # 累加：c_k · M^k |ψ⟩
        accumulated = accumulated + qsvt_polynomial_coefficients[k] * monomial
    
    # 归一化（按系数 L1 范数）
    norm = torch.linalg.vector_norm(qsvt_polynomial_coefficients, ord=1)
    
    return accumulated / norm

> **为什么这样设计？** 经典模拟中直接使用多项式求值的代数公式而非构造 QSVT 电路，避免了相位角度 $\{\phi_k\}$ 的繁琐计算。但对于`degree = 3`（论文默认值），这意味着需要调用 `apply_lcu` 共 `1+2+3 = 6` 次。模型的参数数量不直接增加（只有 `d+1` 个可训练系数），但计算量随 `d` 线性增长。

#### 8.3.4 Quixer 类的完整定义

In [ ]:
class Quixer(torch.nn.Module):
    def __init__(
        self,
        n_qubits: int,                   # 量子比特数（默认 6）
        n_tokens: int,                   # 上下文窗口大小（默认 32）
        qsvt_polynomial_degree: int,     # QSVT 多项式次数（默认 3）
        n_ansatz_layers: int,            # PQC 层数（默认 4）
        vocabulary_size: int,            # 词表大小
        embedding_dimension: int,        # 嵌入维度（默认 512）
        dropout: float,                  # Dropout 率（默认 0.2）
        batch_size: int,                 # 批次大小（默认 20）
        device: torch.device,
    ):
        super().__init__()
        
        # === 参数推导 ===
        n_pqc_parameters = 4 * n_qubits * n_ansatz_layers
        n_polynomial_coefficients = qsvt_polynomial_degree + 1
        
        # === 经典嵌入 + 角度参数化 ===
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension)
        self.embedding_to_angles = nn.Linear(
            embedding_dimension, n_pqc_parameters
        )
        self.dropout = nn.Dropout(dropout)
        
        # === 量子设备（经典模拟） ===
        self.torchquantum_device = tq.QuantumDevice(
            n_wires=n_qubits, bsz=batch_size
        )
        
        # === 令牌 PQC（Ansatz 14） ===
        token_ansatz = ansatz_14_torchquantum_specification(
            n_qubits, n_ansatz_layers
        )
        self.token_pqc = tq.GeneralEncoder(token_ansatz)
        
        # === 可训练参数 ===
        # QSVT 多项式系数（实数）
        self.qsvt_coefficients = nn.Parameter(
            torch.randn(n_polynomial_coefficients)
        )
        # LCU 混合系数（复数！）
        self.lcu_coefficients = nn.Parameter(
            torch.randn(n_tokens, dtype=torch.complex64)
        )
        
        # === 量子前馈网络 ===
        ff_ansatz = ansatz_14_torchquantum_specification(n_qubits, layers=1)
        self.quantum_feedforward = tq.GeneralEncoder(ff_ansatz)
        self.ff_parameters = nn.Parameter(
            torch.randn(1, n_pqc_parameters)
        )
        
        # === 测量 ===
        self.measure_all_x_y_z = tq.MeasureMultipleTimes(
            [list(range(n_qubits))] * 3,  # 三组：X, Y, Z
            ["pauli_x", "pauli_y", "pauli_z"]
        )
        
        # === 经典输出头 ===
        self.output_head = nn.Sequential(
            nn.Linear(3 * n_qubits, embedding_dimension),
            nn.ReLU(),
            nn.Linear(embedding_dimension, vocabulary_size),
        )
    
    def forward(self, x):
        """
        x: [batch_size, n_tokens] — 令牌索引
        返回: (logits, mean_probability)
        """
        batch_size = x.shape[0]
        
        # === 1. LCU 系数准备 ===
        lcu_coeffs = self.lcu_coefficients.repeat(batch_size, 1)
        lcu_coeffs = lcu_coeffs / lcu_coeffs.abs().sum(dim=1, keepdim=True)
        # L1 归一化：Σ |b_j| = 1
        
        # === 2. 经典嵌入 + 角度参数化 ===
        emb = self.embedding(x)              # [B, n_tokens, d_emb]
        angles = self.embedding_to_angles(
            self.dropout(emb)
        )                                    # [B, n_tokens, n_pqc_params]
        
        # === 3. 初始量子态 |0...0⟩ ===
        initial_state = torch.zeros(
            batch_size, 2**self.n_qubits, 
            dtype=torch.complex64, device=x.device
        )
        initial_state[:, 0] = 1.0 + 0.0j
        
        # === 4. QSVT + LCU：核心量子令牌混合 ===
        qsvt_lcu_state = apply_qsvt_and_lcu(
            initial_state, angles, self.token_pqc,
            self.torchquantum_device, self.n_qubits,
            lcu_coeffs, self.qsvt_coefficients,
        )
        
        # === 5. 记录后选择概率 ===
        probabilities = torch.linalg.vector_norm(
            qsvt_lcu_state, dim=-1
        )  # L2 范数 × 1
        
        # === 6. L2 归一化 ===
        qsvt_lcu_state = torch.nn.functional.normalize(
            qsvt_lcu_state, dim=-1
        )
        
        # === 7. 量子前馈网络 ===
        # 将归一化态加载到量子设备
        self.torchquantum_device.reset_batch_size(batch_size)
        self.torchquantum_device.set_states(qsvt_lcu_state)
        
        # 应用单层 PQC
        ff_params = self.ff_parameters.repeat(1, batch_size)
        self.quantum_feedforward(
            self.torchquantum_device, ff_params.float()
        )
        
        # === 8. 测量 ===
        expectation_values = self.measure_all_x_y_z(
            self.torchquantum_device
        )
        # 重整为 [batch_size, 3 * n_qubits]
        measurements = expectation_values \
            .reshape(3, batch_size, self.n_qubits) \
            .moveaxis(0, 1) \
            .reshape(batch_size, -1)
        
        # === 9. 经典输出投影 ===
        logits = self.output_head(measurements)
        
        return logits, probabilities.mean()

### 8.4 模型复杂度分析

| 组件 | 参数量 | 计算复杂度 |
| --- | --- | --- |
| Embedding | $$\mathrm{vocab\_size} \times d_{\mathrm{emb}}$$ | $$O(B \cdot n \cdot d_{\mathrm{emb}})$$ |
| Embedding → Angles | $$d_{\mathrm{emb}} \times (4q n_{\mathrm{layers}})$$ | $$O(B \cdot n \cdot d_{\mathrm{emb}} \cdot 4q n_{\mathrm{layers}})$$ |
| LCU（每步） | $n$（复数系数） | $$O(B \cdot n \cdot 4^q)$$ |
| QSVT（(d) 次） | $d+1$（多项式系数） | $$O(B \cdot d \cdot n \cdot 4^q)$$ |
| Quantum FF | $4q$（角度参数） | $$O(B \cdot 4^q)$$ |
| 测量 | $3q$（期望值读取） | $$O(B \cdot q \cdot 4^q)$$ |
| 输出头 | $$3q \times d_{\mathrm{emb}} + d_{\mathrm{emb}} \times \mathrm{vocab}$$ | $$O(B \cdot 3q \cdot d_{\mathrm{emb}} + B \cdot d_{\mathrm{emb}} \cdot \mathrm{vocab})$$ |

**关键瓶颈**：所有量子操作的计算复杂度都是 $O(4^q)$——这是经典模拟量子电路的根本限制。$q=6$ 时，$4^6 = 4096$（尚可处理）；$q=10$ 时，$4^{10} \approx 10^6$（勉强可处理）；$q=20$ 时，$4^{20} \approx 10^{12}$（完全不可处理）。这是 Quixer 当前限制在 6 个量子比特的根本原因。

---

## 第九章：环境搭建与动手运行

> **目标**：在你的机器上成功安装并运行 Quixer。

### 9.1 环境要求

| 依赖 | 精确版本 | 说明 |
|------|---------|------|
| Python | ≥ 3.11 | **严格要求** |
| PyTorch | 2.3.0（推荐） | 核心深度学习框架 |
| torchtext | 0.18.0 | 文本处理 |
| torchvision | 0.18.0 | 随 PyTorch 安装 |
| torchquantum | ≥ 0.1.8, < 0.2 | 量子电路模拟 |
| Qiskit | < 1.0 | **禁用 1.x 版本**（API 重构） |
| Qiskit Aer | 0.13.3 | 量子模拟后端 |
| Qiskit IBM Provider | 0.10.0 | IBM 量子硬件接口 |
| Qiskit IBM Runtime | 0.20.0 | IBM 量子运行时 |
| datasets | ≥ 3.2, < 3.3 | Hugging Face 数据集 |
| tqdm | ≥ 4.67, < 4.68 | 进度条 |
| CUDA | 可选 | GPU 加速（训练约 100 倍快） |

### 9.2 踩坑预警：版本兼容性问题

Qiskit 1.0 进行了颠覆性的 API 重构（移除了元包架构，引入了 V2 原语）。Quixer 的代码是在 Qiskit < 1.0 的环境下编写的。**如果你直接 `pip install qiskit`（会安装 ≥1.0），Quixer 将无法运行。**

因此强烈建议**创建独立的虚拟环境**。

### 9.3 安装步骤（macOS/Linux）

In [ ]:
#!/usr/bin/env bash
set -euo pipefail

# 1. 进入工作目录
cd /Users/saintway/Downloads/Quixer

# 2. 创建存放克隆仓库的目录
mkdir -p QuixerRepo
cd QuixerRepo

# 3. 克隆 Quixer 仓库
git clone https://github.com/Quantinuum/Quixer.git .

# 4. 创建 Python 3.11 虚拟环境
/opt/homebrew/bin/python3.11 -m venv .venv311

# 5. 激活虚拟环境，并升级 pip/依赖工具
source .venv311/bin/activate
python -m pip install --upgrade pip setuptools wheel

# 6. 安装 PyTorch 2.3.0 及配套包（CPU 版本）
python -m pip install torch==2.3.0 torchvision==0.18.0 torchtext==0.18.0 \
  --index-url https://download.pytorch.org/whl/cpu

# 7. 安装 Quixer 所需的量子和数据包依赖
python -m pip install 'torchquantum>=0.1.8,<0.2' 'qiskit<1.0' \
  'qiskit-aer==0.13.3' 'qiskit-ibm-provider==0.10.0' \
  'qiskit-ibm-runtime==0.20.0' 'datasets>=3.2,<3.3' 'tqdm>=4.67,<4.68'

# 8. （可选）验证关键包是否可导入
python - <<'PY'
import torch, torchtext, torchquantum, qiskit
print("torch", torch.__version__)
print("torchtext", torchtext.__version__)
print("torchquantum", torchquantum.__version__)
print("qiskit", qiskit.__version__)
PY

# 9. 修改 run.py 的 epochs 为 5
python - <<'PY'
from pathlib import Path
path = Path("run.py")
text = path.read_text()
text = text.replace('"epochs": 30', '"epochs": 5', 1)
text = text.replace('"epochs": 30', '"epochs": 5', 1)
text = text.replace('"epochs": 30', '"epochs": 5', 1)
text = text.replace('"epochs": 30', '"epochs": 5', 1)
path.write_text(text)
PY

# 10. 在 quixer/quixer_model.py 增加调试 print（示例，可根据需要手工修改）
python - <<'PY'
from pathlib import Path
path = Path("quixer/quixer_model.py")
text = path.read_text()
text = text.replace(
    "    accumulated_state = qsvt_polynomial_coefficients[0] * initial_states\n",
    "    print(f\"[Quixer] apply_qsvt_and_lcu initial_states={initial_states.shape} "
    "pqc_parameters={pqc_parameters.shape} lcu_coefficients={lcu_coefficients.shape} "
    "qsvt_polynomial_coefficients={qsvt_polynomial_coefficients.shape}\")\n"
    "    accumulated_state = qsvt_polynomial_coefficients[0] * initial_states\n"
)
text = text.replace(
    "    lcu_applied_to_state = torch.einsum(\"bwi,bw->bi\", states, lcu_coefficients)\n",
    "    print(f\"[Quixer] apply_lcu states={states.shape}\")\n"
    "    lcu_applied_to_state = torch.einsum(\"bwi,bw->bi\", states, lcu_coefficients)\n"
    "    print(f\"[Quixer] apply_lcu lcu_applied_to_state={lcu_applied_to_state.shape}\")\n"
)
path.write_text(text)
PY

echo "QuixerRepo 创建完成，虚拟环境在 .venv311"

```bash
# === 步骤 1：创建虚拟环境 ===
conda create -n quixer python=3.11 -y
conda activate quixer

# === 步骤 2：安装 PyTorch 2.3.0（CPU 版本） ===
conda install pytorch==2.3.0 torchvision==0.18.0 \
    torchtext==0.18.0 torchaudio==2.3.0 cpuonly -c pytorch -y

# 或者 CUDA 版本（如果有 NVIDIA GPU）：
# conda install pytorch==2.3.0 torchvision==0.18.0 \
#     torchtext==0.18.0 torchaudio==2.3.0 pytorch-cuda=12.1 -c pytorch -y

# === 步骤 3：克隆仓库 ===
git clone https://github.com/Quantinuum/Quixer.git
cd Quixer

# === 步骤 4：安装 Quixer 及其依赖 ===
pip install -e .

# === 步骤 5：验证安装 ===
python -c "import torch; import qiskit; import torchquantum; \
    print(f'PyTorch {torch.__version__}'); \
    print(f'Qiskit {qiskit.__version__}'); \
    print('安装成功!')"
```

**常见错误与解决方案**：

| 错误 | 原因 | 解决 |
|------|------|------|
| `ModuleNotFoundError: No module named 'qiskit.providers'` | Qiskit 1.0+ 移除了旧 API | 降级到 `pip install qiskit<1.0` |
| `RuntimeError: CUDA out of memory` | GPU 显存不足 | 减小 batch_size 或用 `-d cpu` |
| `ImportError: cannot import name 'get_tokenizer'` | torchtext 版本不对 | 确保 `torchtext==0.18.0` |
| `AttributeError: 'QuantumDevice' object has no attribute 'set_states'` | torchquantum 版本问题 | 确保 `torchquantum>=0.1.8,<0.2` |

### 9.4 运行训练

In [ ]:
cd QuixerRepo

In [ ]:
source .venv311/bin/activate
python run.py -d cpu -m Quixer

In [ ]:
./.venv311/bin/python run.py -d cpu -m Quixer

```bash
# === 仅运行 Quixer（CPU） ===
python run.py -d cpu -m Quixer

# === 运行 Quixer + 经典基线（CPU） ===
python run.py -d cpu -m Quixer Transformer LSTM FNet

# === GPU 版本（如果有 CUDA） ===
python run.py -d cuda -m Quixer

# === 仅运行经典基线（用于对比） ===
python run.py -d cpu -m Transformer LSTM FNet
```

### 9.5 训练过程详解

训练过程将：

1. **下载数据**：自动从 Hugging Face Hub 下载 Penn Treebank 数据集
2. **构建词表**：使用 `basic_english` 分词器，构建包含 `<pad>`、`<unk>`、`<eos>` 特殊令牌的词表
3. **批量化**：将数据按 `window=32` 的上下文窗口组织为批次
4. **多轮训练**：默认 40 个 epoch，使用 Adam 优化器，Cosine Annealing 学习率调度
5. **自动保存**：将每个 epoch 的最优模型保存到 `./trained_models/` 目录
6. **多种子评估**：对每个配置运行 10 个不同的随机种子

单次完整训练（1 个模型 + 1 个嵌入维度 + 10 个种子 + 40 epochs）在 CPU 上大约需要 **6-12 小时**，在 GPU 上大约需要 **10-20 分钟**。

### 9.6 训练过程中的输出解读

```
Epoch: 01 | Time: 2m 30s
    Train Loss: 5.832 | Train ppl: 341.2
     Val. Loss: 5.651 |  Val. ppl: 284.5

Epoch: 10 | Time: 2m 15s
    Train Loss: 4.912 | Train ppl: 135.8
     Val. Loss: 5.012 |  Val. ppl: 149.9
...
FINAL TRAINED MODEL STATS:
     Val. Loss: 4.897 |  Val. ppl: 133.8
     Test Loss: 4.912 |  Test ppl: 135.7
```

其中：
- **Loss**：Cross-Entropy Loss（越低越好）
- **ppl（Perplexity，困惑度）**：$\text{ppl} = e^{\text{loss}}$，衡量模型对下一令牌预测的不确定性。困惑度越低，模型越『自信』且准确。作为参考：随机猜测的困惑度 ≈ 词表大小（Penn Treebank 约 10,000）

### 9.7 实验设计解读

run.py 中 Quixer 和经典基线的超参数：

| 参数 | Quixer | Transformer | LSTM | FNet |
|------|--------|-------------|------|------|
| 嵌入维度 | **512** | 96, 128 | 96, 128 | 96, 128 |
| 层数 | 3 | 3 | 3 | 3 |
| 窗口 | 32 | 32 | 32 | 32 |
| Epochs | 40 | 40 | 40 | 40 |
| Batch | 20 | 20 | 20 | 20 |
| Dropout | 0.2 | 0.2 | 0.2 | 0.2 |
| 量子比特 | 6 | N/A | N/A | N/A |
| Ansatz 层数 | 4 | N/A | N/A | N/A |

**为什么 Quixer 的嵌入维度高达 512？**

Quixer 不使用多头注意力，其『表达能力』主要来自：(1) PQC 在 $2^6=64$ 维希尔伯特空间中的演化；(2) 经典嵌入向高维角度参数的线性映射。需要 $d_{\text{emb}} \to n_{\text{pqc\_params}} = 4 \times 6 \times 4 = 96$ 的映射。512 维嵌入提供了充足的过完备表示能力。经典 Transformer 用 128 维嵌入则因为多头注意力提供了足够的交互表达能力。

---

## 第十章：实验结果与性能分析

> **目标**：理解 Quixer 目前的性能水平及其含义。

### 10.1 Penn Treebank 结果

Quixer 在 Penn Treebank 语言建模任务上的表现（论文表 1）：

| 模型 | 测试困惑度 (Test PPL) | 参数量 |
|------|---------------------|--------|
| LSTM (128d) | ~100-110 | ~6M |
| FNet (128d) | ~110-120 | ~1M |
| Transformer (128d) | ~95-105 | ~4M |
| **Quixer (512d)** | **~135-145** | ~10M |

> **谨慎解读**：论文声称 Quixer 表现『competitive with an equivalent classical baseline』（与等效的经典基线有竞争力）。但这一声明在对抗验证中以 0-3 被否定——三个独立验证者均无法确认 Quixer 在相同参数量条件下接近经典基线的性能。Quixer 的实际表现**显著弱于**同等参数规模的经典 Transformer。

### 10.2 为什么 Quixer 不如经典模型？

这涉及几个根本原因：

1. **量子态矢量维度限制**：$q=6$ 意味着量子态只有 64 维，远小于经典 Transformer 的隐藏维度。即使有 512 维的嵌入『外衣』，真正进行令牌混合的空间只有 64 维。

2. **梯度消失问题（Barren Plateau）**：参数化量子电路在随机初始化时，梯度指数级趋近于零。这是量子机器学习的根本性挑战——Quixer 无法幸免。原论文第 6 节将此明确列为开放问题。

3. **有限上下文**：窗口仅 32 个令牌——后续研究 QMamba 指出这是 Quixer 的一大瓶颈。

4. **经典模拟的近似**：经典模拟中 QSVT 通过直接计算 $M^k|0\rangle$ 来实现，免去了构造 SELECT-PREPARE 电路和计算相位角度 $\{\phi_k\}$ 的困难。但这意味着，经典模拟并不能完全代表真实量子硬件上的行为（后者可能因为干涉效应产生更多表达力，也可能因为噪声产生更差表现）。

### 10.3 Aalto 大学复现结果

2024 年 9 月，Aalto 大学的一篇学士论文在经典模拟环境下独立复现了 Quixer 实验。关键发现：

- Quixer 可以在 Penn Treebank 上训练并获得合理的结果
- 训练极其缓慢（CPU 上需要数小时到数天）
- 性能低于经典 Transformer 基线
- 论文确认了梯度消失问题的存在

### 10.4 性能评估的正确视角

评估 Quixer 的性能时，以下几点非常重要：

1. **概念验证 > 性能竞赛**：Quixer 的价值在于**证明了量子原语可以构建可训练的语言模型**，而不是与 GPT-4 竞争。

2. **经典模拟 ≠ 量子实现**：当前所有 Quixer 结果都来自经典模拟，其计算瓶颈（$O(4^q)$）完全不同于真实量子硬件的瓶颈。经典模拟中表现一般的模型在量子硬件上可能有完全不同的行为。

3. **第一代产品**：Quixer 是量子 Transformer 领域的开创性工作。第一个晶体管计算机也无法与现代 CPU 比性能。

4. **框架价值**：论文强调 Quixer 是一个**框架**——其参数化组件可以被替换为不同结构，从而产生一类全新的量子 Transformer。比较『单个实例』的性能是不全面的。

---

## 第十一章：局限性与开放问题

> **目标**：诚实评估 Quixer 当前的局限性，识别值得关注的开放问题。

### 11.1 已确认的局限性

#### 11.1.1 梯度消失（Vanishing Gradients）

这是 Quixer 最根本的挑战。参数化量子电路（PQC）普遍存在 **Barren Plateau 现象**：当电路深度或量子比特数增加时，梯度以指数方式趋近于零。

具体到 Quixer：
- Ansatz 14 使用 4 层，在 6 量子比特下仍然可训练
- 但扩展到更多量子比特或更深电路时，梯度消失将变得严重
- 这限制了模型规模的扩展

**原论文原文**（第 6 节）：

> "Finding an instance of Quixer that does not suffer from vanishing gradients while being too expressive for classical simulation is left to future work."

翻译：**找到一个既无梯度消失问题、又超出经典模拟能力的 Quixer 实例，是留给未来工作的开放问题。**

#### 11.1.2 经典可模拟性（Classical Simulability）

当前所有 Quixer 实验使用的量子电路（6 量子比特、4 层 PQC、3 次 QSVT）在经典计算机上可以精确模拟。这意味着：

- **没有任何量子优势**——经典计算机完全可以模拟这些电路
- 要证明量子优势，需要扩大到经典不可模拟的规模
- 但扩大规模又面临梯度消失问题——**两难困境**

#### 11.1.3 有限上下文窗口（32 令牌）

在 Penn Treebank 实验中，Quixer 的上下文窗口仅为 32 个令牌。这远小于现代 LLM 的数万到数十万令牌上下文。后续工作 QMamba（ICAART 2025）以 Quixer 为对比基线，指出这个限制是其核心瓶颈之一。

#### 11.1.4 对经典输出头的强依赖

当前模型最后的『经典输出头』（两层 MLP，从 3q 维到 vocab_size 维）承担了大量的计算工作。没有这个输出头，模型的性能会大幅下降。这引发了一个问题：量子部分到底学到了多少，还是经典 MLP 在做大部分『重活』？

#### 11.1.5 未解决的量子硬件部署

- 论文中的资源估算（§3.5）考虑了后选择概率和门复杂度，但实际硬件部署尚未被独立验证
- Quantinuum 2025 年 6 月的博客宣称已将 Quixer 部署到真实硬件，但缺少性能数据和第三方验证
- 后选择导致的 $O(1/p_{\text{success}})$ 采样开销在实际中可能非常高

### 11.2 关键开放问题

这些问题不仅对 Quixer 重要，对整个量子机器学习领域也至关重要：

1. **量子优势边界**：量子比特数、电路深度、QSVT 多项式次数——需要达到什么规模的组合，才能在语言建模任务上产生可验证的量子优势？

2. **梯度消失的解决**：是否存在某种电路结构或初始化策略，能从根本上避免 PQC 的 Barren Plateau？

3. **LCU 系数的可解释性**：$b_j = e^{i\gamma_j}|a_j|^2$ 究竟代表了什么语义？能否像经典注意力的 softmax 权重那样被可视化和解读？

4. **与经典架构的融合**：Quixer 使用纯粹的 LCU+QSVT 替代注意力——但如果采用混合策略（部分量子、部分经典）会更好吗？

5. **扩展到更大模型**：能否设计一种『量子蒸馏』方案，用小规模量子 Transformer 指导大规模经典 Transformer 训练？

---

## 第十二章：后续研究方向与生态

> **目标**：了解 Quixer 启发了哪些后续工作，以及你可以在哪些方向上贡献。

### 12.1 QMamba：从 Transformer 到状态空间模型

QMamba（ICAART 2025）是 Quixer 最直接的后续工作。它基于以下观察：

- Quixer 在 Penn Treebank 上的上下文窗口仅 32 个令牌
- Mamba（选择性状态空间模型）在长序列建模上表现出色
- 将 Mamba 的选择性 SSM 机制移植到量子域可能克服 Transformer 的上下文限制

QMamba 是**首个量子 Mamba 模型**，以 Quixer 为对比基线，代表了量子序列建模从 Transformer 范式向 SSM 范式的转移。

### 12.2 Quixer 框架的可扩展性

原论文第 5 节特别强调了 Quixer 作为**框架**的价值：

> "its parameterised components can be substituted with fixed structures to yield new classes of quantum transformers."

翻译：**Quixer 的参数化组件可以被替换为固定结构，从而产生全新类别的量子 Transformer。**

具体来说，你可以在以下组件上做文章：

| 组件 | 当前选择 | 替代方案 |
|------|---------|---------|
| 酉嵌入电路 | Ansatz 14 | 其他 PQC ansatz、Hardware-Efficient Ansatz |
| LCU 系数结构 | 可训练复数权重 | 固定权重（如相等权重、基于令牌距离的衰减权重） |
| QSVT 多项式 | 通用多项式 | 特定函数的多项式近似（如 sigmoid、tanh） |
| 测量方案 | X, Y, Z 全测量 | 稀疏测量、自适应测量、基于任务的测量选择 |

### 12.3 相关项目与工具链

| 项目 | 说明 | 链接 |
|------|------|------|
| TorchQuantum | MIT 开发的量子机器学习框架，Quixer 的核心依赖 | github.com/mit-han-lab/torchquantum |
| PennyLane | Xanadu 的量子机器学习库，支持自动微分和 QSVT | pennylane.ai |
| Qiskit | IBM 的量子计算框架（注意版本 < 1.0） | qiskit.org |
| Lambeq | Quantinuum 的量子 NLP 框架 | github.com/Quantinuum/lambeq |
| pyqsp | QSVT 相位角度计算工具 | github.com/ichuang/pyqsp |
| QSPPACK | QSVT 的 MATLAB 实现 | github.com/qsppack/QSPPACK |

### 12.4 入门的动手实验建议

如果你想在 Quixer 的基础上做研究和实验，以下是一条推荐的路径：

**阶段 1：理解代码**

1. 成功安装并运行 Quixer（按照第九章）
2. 在 `quixer_model.py` 中添加 `print` 语句，观察每个步骤的 Tensor Shape
3. 修改 `run.py` 中的超参数（如减少 epochs 到 5，观察训练过程）

**阶段 2：小改动**

4. 修改 LCU 系数初始化方式（如改为均匀分布）
5. 改变 QSVT 多项式次数（改为 1 或 5，观察性能变化）
6. 将 `embedding_dimension` 从 512 改为 256，观察效果

**阶段 3：组件替换**

7. 用不同的 PQC ansatz 替换 Ansatz 14
8. 尝试不同的测量方案（如只测量 Z 期望值）
9. 实现一个无需输出头的『纯量子』版本

**阶段 4：新模型设计**

10. 参考 Quixer 框架设计你自己的量子 Transformer 变体
11. 探索 LCU+QSVT 的组合在分类、序列标注等任务上的应用
12. 研究混合量子-经典架构：用 Quixer 作为『注意力替代』插入经典 Transformer

### 12.5 追踪最新进展

由于该领域发展迅速，建议关注以下渠道：

- arXiv 量子物理学（quant-ph）和机器学习（cs.LG）子类别
- Quantinuum 官方博客及研究页面
- 量子机器学习会议：QML（Quantum Machine Learning）、QTML（Quantum Techniques in Machine Learning）
- ICAART（International Conference on Agents and Artificial Intelligence）— QMamba 发表于此

---

## 附录 A：术语对照表

| 英文 | 中文 | 说明 |
|------|------|------|
| Ansatz | 拟设 / 电路模板 | 参数化量子电路的固定结构 |
| Barren Plateau | 贫瘠高原 | 梯度指数衰减到零的现象 |
| Block Encoding | 块编码 | 用酉矩阵的子块表示任意矩阵 |
| Expectation Value | 期望值 | $\langle \psi \lvert O \rvert \psi \rangle$，可观测量的平均测量结果 |
| Feed-Forward Network (FFN) | 前馈网络 | 逐位置的经典非线性变换 |
| Linear Combination of Unitaries (LCU) | 线性酉组合 | $\sum_j b_j U_j$ |
| Pauli Matrices (X, Y, Z) | 泡利矩阵 | 基本量子门/可观测量 |
| Penn Treebank (PTB) | 宾州树库 | 经典语言建模基准数据集 |
| Perplexity (PPL) | 困惑度 | $\exp(\text{cross-entropy loss})$ |
| Postselection | 后选择 | 仅保留满足条件的测量结果 |
| Quantum Singular Value Transform (QSVT) | 量子奇异值变换 | 对块编码矩阵施加多项式变换 |
| SELECT-PREPARE | 选择-准备电路 | LCU 的标准电路实现 |
| Token Mixing | 令牌混合 | 不同令牌表示之间交换信息 |
| Unitary | 酉（矩阵/变换） | 满足 $U^\dagger U = I$ 的复矩阵 |

## 附录 B：关键公式汇总

| 公式 | 编号 | 说明 |
|------|------|------|
| $$\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$$ | (1) | 经典点积自注意力 |
| $$M = \sum_{j=0}^{n-1} b_j U_j$$ | (2) | LCU 令牌混合 |
| $$U_{\text{SEL}} = \sum_j \|j\rangle\langle j\| \otimes U_j$$ | (3) | SELECT 酉变换 |
| $$U_{\text{PREP}}\|0\rangle = \sum_j a_j \|j\rangle$$ | (4) | PREPARE 酉变换 |
| $$U_M = (U_{\text{PREP}}^\dagger \otimes I) U_{\text{SEL}} (U_{\text{PREP}} \otimes I)$$ | (5) | LCU 完整电路 |
| $$M = (\langle 0\| \otimes I) U_M (\|0\rangle \otimes I)$$ | (6) | 块编码投影 |
| $$P_c(M) = \sum_{k=0}^d c_k M^k$$ | (7) | QSVT 多项式变换 |
| $$\|P_c(x)\| \leq 1, \forall x \in [-1,1]$$ | (8) | QSVT 有界性约束 |
| $$\text{parity}(P_c) = d \bmod 2$$ | (9) | QSVT 奇偶性约束 |

## 附录 C：推荐阅读路径

### 如果你只想快速了解
1. 第三章：Quixer 宏观概览（15 分钟）
2. 第七章：Quixer 完整架构详解（40 分钟）

### 如果你想深入理解原理
1. 第一章：GPT/Transformer 回顾
2. 第四章：块编码
3. 第五章：LCU
4. 第六章：QSVT
5. 第七章：完整架构

### 如果你想动手实现
1. 直接跳到第九章：环境搭建与动手运行
2. 遇到不理解的地方回查对应章节
3. 配合第八章源码解析阅读代码

### 如果你想做研究
1. 完整阅读所有章节
2. 仔细阅读原论文 arXiv:2406.04305
3. 研究第十一章的开放问题，选择你感兴趣的方向
4. 阅读 QMamba 论文了解最新进展

---

## 参考资料

1. Khatri, N., Matos, G., Coopmans, L., & Clark, S. (2024). **Quixer: A Quantum Transformer Model**. arXiv:2406.04305. https://arxiv.org/abs/2406.04305
2. Quixer 官方代码库. https://github.com/Quantinuum/Quixer
3. Quantinuum Blog. "Announcing Quixer — Quantinuum's State-of-the-Art Quantum Transformer". https://www.quantinuum.com/blog/announcing-quixer
4. Quantinuum Blog. "Our Hardware is Now Running Quantum Transformers". https://www.quantinuum.com/blog/our-hardware-is-now-running-quantum-transformers
5. Vaswani, A., et al. (2017). **Attention Is All You Need**. NeurIPS 2017.
6. Lee-Thorp, J., et al. (2022). **FNet: Mixing Tokens with Fourier Transforms**. NAACL 2022.
7. Stenzel, L., & Kölle, M. (2025). **QMamba: Quantum Selective State Space Models for Language Modeling**. ICAART 2025.
8. Sim, S., et al. (2019). **Expressive power of parameterized quantum circuits**. arXiv:1905.10876. （Ansatz 14 的来源）
9. Gilyén, A., et al. (2019). **Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics**. STOC 2019. （QSVT 的原始论文）
10. Childs, A. M., & Wiebe, N. (2012). **Hamiltonian simulation using linear combinations of unitary operations**. QIC 2012. （LCU 的原始论文）
11. Aalto 大学学士论文 (2024). **Evaluation of the Quixer Quantum Transformer Model**. https://aaltodoc.aalto.fi/items/a3998701-7b17-48e2-a74e-bcf87d927c26/full
12. TorchQuantum. https://github.com/mit-han-lab/torchquantum
13. PennyLane QSVT 文档. https://pennylane.ai

---

> **教程版本**：v1.0（2026 年 5 月）
>
> **基于工作流**：deep-research（94 个搜索/验证代理，1,928,809 tokens）
>
> **声明**：本教程基于对 Quixer 论文、源代码和第三方资料的深入研究编写。所有核心声明均经过 3 票对抗验证（共验证 25 项声明，确认 13 项，否定 12 项）。教程中的代码解析基于 Apache 2.0 许可证下的官方开源实现。
>
> **致谢**：感谢 Quantinuum 团队开源 Quixer 代码和数据，使社区能够学习、复现和在此基础上继续创新。